In [1]:
!pip install psutil

In [2]:
# collect_sysinfo.py ; output in json format
import json
import psutil
import platform
import socket
import datetime

now = datetime.datetime.now()
print(now)

def collect_system_info():
    data = {}

    # CPU info
    data["cpu"] = {
        "platform": platform.platform(),
        "processor": platform.processor(),
        "physical_cores": psutil.cpu_count(logical=False),
        "logical_processors": psutil.cpu_count(logical=True),
        "cpu_freq": psutil.cpu_freq()._asdict() if psutil.cpu_freq() else None,
        "cpu_percent_total": psutil.cpu_percent(interval=0.1),
        "cpu_percent_per_cpu": psutil.cpu_percent(interval=0.1, percpu=True),
    }

    # Memory info
    data["memory"] = psutil.virtual_memory()._asdict()
    data["swap"] = psutil.swap_memory()._asdict()

    # Storage info (toutes les partitions)
    storage = {}
    for part in psutil.disk_partitions(all=False):
        try:
            usage = psutil.disk_usage(part.mountpoint)
            storage[part.mountpoint] = {
                "device": part.device,
                "fstype": part.fstype,
                "opts": part.opts,
                "total": usage.total,
                "used": usage.used,
                "free": usage.free,
                "percent": usage.percent,
            }
        except PermissionError:
            continue
    data["storage"] = storage

    # Network info
    net = {}
    addrs = psutil.net_if_addrs()
    stats = psutil.net_if_stats()
    io = psutil.net_io_counters(pernic=True)
    for iface, addr_list in addrs.items():
        net[iface] = {
            "addrs": [a._asdict() for a in addr_list],
            "stats": stats[iface]._asdict() if iface in stats else None,
            "io": io[iface]._asdict() if iface in io else None,
        }
    data["network"] = {
        "hostname": socket.gethostname(),
        "fqdn": socket.getfqdn(),
        "interfaces": net,
    }
    return data

if __name__ == "__main__":
    info = collect_system_info()
    print(json.dumps(info, indent=2))

2026-10-13 15:31:53.460683
{
  "cpu": {
    "platform": "Windows-11-10.0.26100-SP0",
    "processor": "AMD64 Family 23 Model 104 Stepping 1, AuthenticAMD",
    "physical_cores": 8,
    "logical_processors": 16,
    "cpu_freq": {
      "current": 1801.0,
      "min": 0.0,
      "max": 1801.0
    },
    "cpu_percent_total": 24.1,
    "cpu_percent_per_cpu": [
      16.7,
      0.0,
      83.3,
      28.6,
      16.7,
      0.0,
      33.3,
      0.0,
      0.0,
      0.0,
      0.0,
      14.3,
      50.0,
      28.6,
      0.0,
      0.0
    ]
  },
  "memory": {
    "total": 15915737088,
    "available": 2928709632,
    "percent": 81.6,
    "used": 12987027456,
    "free": 2928709632
  },
  "swap": {
    "total": 11181338624,
    "used": 536596480,
    "free": 10644742144,
    "percent": 4.8,
    "sin": 0,
    "sout": 0
  },
  "storage": {
    "C:\\": {
      "device": "C:\\",
      "fstype": "NTFS",
      "opts": "rw,fixed",
      "total": 509722226688,
      "used": 503868788736,
     